### 核心公式

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V
$$

三个矩阵都来自同一个输入序列 $X \in \mathbb{R}^{n \times d}$：

$$
Q = XW^Q,\quad K = XW^K,\quad V = XW^V
$$

其中 $W^Q, W^K, W^V \in \mathbb{R}^{d \times d_k}$ 是可学习参数。

逐步拆解  
第一步：计算注意力分数（Attention Scores）
$$
S = QK^T
$$

$S_{ij}$ 就是位置 $i$ 的 Query 和位置 $j$ 的 Key 的点积。直觉：Query 是"我在找什么"，Key 是"我有什么"，点积衡量匹配程度。$S_{ij}$ 越大，说明位置 $j$ 对位置 $i$ 越相关。

第二步：缩放（Scale）
$$
\frac{S}{\sqrt{d_k}}
$$

点积的方差随维度 $d_k$ 增大而增大。如果不缩放，大维度下 softmax 的输入值会很大，梯度趋近于 0（softmax 饱和区）。除以 $\sqrt{d_k}$ 把方差控制回 1。

第三步：Softmax 归一化
$$
A = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)
$$

每一行变成一个概率分布——位置 $i$ 对所有位置的"注意力权重"，加起来等于 1。

第四步：加权聚合
$$
\text{Output} = AV
$$

每个位置的输出 = 所有位置的 Value 向量的加权和，权重由注意力分布决定。直觉：Value 是"实际内容"，注意力权重决定"各拿多少"。

一个具体例子
假设句子是 "The cat sat on the mat"，我们要计算 "sat" 的新表示：  

每个词都算出 Q, K, V  
"sat" 的 Q 和所有词的 K 做点积 → 得到 6 个分数  
缩放 + softmax → 比如 "sat" 对自己 0.4，"cat" 0.3，"mat" 0.2，其他接近 0  
用这些权重对所有词的 V 加权求和 → "sat" 的新表示融合了 "谁发出了这个动作"（cat）和 "在哪里"（mat）的信息      
#### 多头注意力（Multi-Head Attention）  
单头只能学一种"关注模式"。多头 = 并行做多次自注意力，每次用不同的 $W^Q, W^K, W^V$，然后把结果拼起来再投影：  

$$
\text{MultiHead}(X) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)W^O
$$

$$
\text{head}_i = \text{Attention}(XW^Q_i, XW^K_i, XW^V_i)
$$

不同头学会关注不同的关系：一个头关注语法依赖，一个头关注语义相似，一个头关注位置邻近，等等。

## 代码实现

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# ============================================================
# 1. 缩放点积注意力 — 自注意力的核心
# ============================================================
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q, K, V: (batch批量大小, n_heads注意力头数, seq_len序列长度, d_k每个头的维度)
    mask:    (batch, 1, seq_len, seq_len) 或 (seq_len, seq_len)
              True/1 的位置会被 mask 掉（设为 -inf）
    """
    d_k = Q.shape[-1]

    # ① 计算注意力分数: Q @ K^T  (batch, heads, seq, seq)
    scores = Q @ K.transpose(-2, -1)          # 把最后两维交换 

    # ② 缩放
    scores = scores / (d_k ** 0.5)

    # ③ mask（可选），如 decoder 中遮住未来位置
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    # ④ softmax 得到注意力权重
    attn_weights = F.softmax(scores, dim=-1)

    # ⑤ 加权聚合 V
    output = attn_weights @ V                 # (batch, heads, seq, d_k)

    return output, attn_weights

In [ ]:
# ============================================================
# 2. 多头自注意力 — 逐行解释
# ============================================================

class MultiHeadSelfAttention(nn.Module):
    """
    把 d_model 维的每个 token 表示，用 n_heads 个不同的"视角"并行做注意力。
    8 个头可能分别学到：语法依赖、语义相似、指代关系、位置邻近……
    最后把各头的输出拼起来投影回 d_model。
    """
    def __init__(self, d_model=512, n_heads=8):
        # ── ① 继承 nn.Module 的初始化 ──
        super().__init__()
        # 必须调用父类 __init__，否则参数注册、梯度计算等 Module 机制不生效

        # ── ② d_model 必须能被 n_heads 整除 ──
        assert d_model % n_heads == 0, \
            "d_model 必须能被 n_heads 整除"
        # 否则拆不出等长的多头。512 / 8 = 64 ✓   500 / 8 ✗

        # ── ③ 记录为实例属性，forward 里要用 ──
        self.d_model = d_model          # 模型总维度，例如 512
        self.n_heads = n_heads          # 注意力头数，例如 8
        self.d_k = d_model // n_heads   # 每个头分到的维度 = 512/8 = 64

        # ── ④ Q、K、V 的投影矩阵（三合一，算完再拆，省掉两个 Linear）──
        # Linear(d_model → 3*d_model): 输出长度是输入的 3 倍
        # 前 1/3 给 Q，中间 1/3 给 K，后 1/3 给 V
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)

        # ── ⑤ 输出投影矩阵 ──
        # 8 个头拼回 d_model 维后，再做一次线性变换得到最终输出
        self.W_o = nn.Linear(d_model, d_model, bias=False)


    # ================================================================
    # split_heads: 把最后一个维度拆成 (n_heads, d_k)，然后把 head 维提前
    # (batch, seq, d_model)  →  (batch, n_heads, seq, d_k)
    # ================================================================
    def split_heads(self, x):
        B, S, _ = x.shape                        # B=batch, S=seq_len

        # view: 换形状不换数据。把 512 拆成 8×64
        x = x.view(B, S, self.n_heads, self.d_k) # (B, S, 8, 64)

        # permute: 按指定顺序重排维度
        # (0,2,1,3): batch(0)不动, head(2)提前, seq(1)后移, d_k(3)不动
        return x.permute(0, 2, 1, 3)             # (B, 8, S, 64)
        # 为什么要换？后面 Q @ K^T 矩阵乘法用最后两维。
        # head 放第二维后，每个头的 (S, 64) 就在最后两维，可以并行算。


    # ================================================================
    # combine_heads: split_heads 的逆操作
    # (batch, n_heads, seq, d_k)  →  (batch, seq, d_model)
    # ================================================================
    def combine_heads(self, x):
        B, _, S, _ = x.shape                     # _ 是 n_heads，已知不用

        # permute 把 head 维塞回第三位: (B, 8, S, 64) → (B, S, 8, 64)
        x = x.permute(0, 2, 1, 3)
        # 此时: dim0=batch, dim1=seq, dim2=head, dim3=d_k

        # contiguous(): 让内存变连续。
        # permute 只改了"怎么看"，底层内存排布没变。
        # 不连续的 tensor 不能 .view()，必须先用 contiguous() 重新排布内存。
        x = x.contiguous()

        # view 把 (B, S, 8, 64) 拼回 (B, S, 512)
        return x.view(B, S, self.d_model)


    # ================================================================
    # forward: 一次前向传播的完整流程
    # x 形状: (batch, seq_len, d_model)
    # ================================================================
    def forward(self, x, mask=None):
        B, S, _ = x.shape                        # 取出 batch 和 seq_len

        # ── 第一步：线性投影算出 Q、K、V ──
        # W_qkv(x): Linear(512→1536)，一次矩阵乘法得到拼接的 qkv
        qkv = self.W_qkv(x)                      # (B, S, 1536)
        # chunk(3, dim=-1): 沿最后一维均匀切成 3 块
        # 前 512 给 Q，中间 512 给 K，后 512 给 V
        Q, K, V = qkv.chunk(3, dim=-1)           # 各 (B, S, 512)

        # ── 第二步：拆成多头 ──
        Q = self.split_heads(Q)                  # (B, 8, S, 64)
        K = self.split_heads(K)                  # (B, 8, S, 64)
        V = self.split_heads(V)                  # (B, 8, S, 64)

        # ── 第三步：核心 — 缩放点积注意力 ──
        # 内部: scores=Q@K^T → /√d_k → softmax → @V
        # attn_out:    (B, 8, S, 64)  — 每个位置加权聚合后的新表示
        # attn_weights:(B, 8, S, S)   — 注意力权重（每行和=1）
        attn_out, attn_weights = scaled_dot_product_attention(Q, K, V, mask)

        # ── 第四步：多头拼接 + 输出投影 ──
        out = self.combine_heads(attn_out)       # (B, S, 512)
        return self.W_o(out), attn_weights       # 最终输出 + 注意力权重（调试/可视化用）


# ================================================================
# 整个数据流的形状变化一览：
# ================================================================
# x            (B, S, d_model)        eg. (2, 6, 512)
#   ↓ W_qkv: Linear(512→1536)
# qkv          (B, S, 3*d_model)      eg. (2, 6, 1536)
#   ↓ chunk(3)
# Q, K, V      (B, S, d_model)        eg. (2, 6, 512)
#   ↓ split_heads: view + permute
# Q, K, V      (B, n_heads, S, d_k)   eg. (2, 8, 6, 64)
#   ↓ scaled_dot_product_attention
#   │  scores = Q @ K^T       → (B, n_heads, S, S)   [S×S 注意力分数矩阵]
#   │  scores = scores / √d_k → (B, n_heads, S, S)   [缩放，防 softmax 饱和]
#   │  attn   = softmax(scores) → (B, n_heads, S, S)  [行归一化，每行=概率分布]
#   │  out    = attn @ V       → (B, n_heads, S, d_k) [用权重聚合 Value]
# attn_out     (B, n_heads, S, d_k)   eg. (2, 8, 6, 64)
#   ↓ combine_heads: permute + contiguous + view
# out          (B, S, d_model)        eg. (2, 6, 512)
#   ↓ W_o: Linear(512→512)
# final        (B, S, d_model)        eg. (2, 6, 512)  [形状与输入完全一致]

In [ ]:
# ============================================================
# 3. 跑一下看看
# ============================================================
d_model, n_heads, seq_len = 512, 8, 6
batch = 2

mha = MultiHeadSelfAttention(d_model, n_heads)
x = torch.randn(batch, seq_len, d_model)        # 2 句话，每句 6 个词，每个词 512 维

out, weights = mha(x)

print(f"输入形状: {x.shape}")
print(f"输出形状: {out.shape}")                  # 和输入一样
print(f"注意力权重形状: {weights.shape}")         # (batch, heads, seq, seq)
print(f"\n每个头的权重求和应为 1: {weights[0, 0, 0].sum().item():.6f}")
print(f"\n注意力热力图 (第 0 个 batch, 第 0 个头):")
print(weights[0, 0].detach().round(decimals=3))

---

## Transformer 完整实现

Transformer = Encoder + Decoder，每个都由多个相同的层堆叠而成：

- **Encoder**: Self-Attention → Add&Norm → FFN → Add&Norm
- **Decoder**: Masked Self-Attention → Add&Norm → Cross-Attention → Add&Norm → FFN → Add&Norm

还需要 **位置编码**，因为注意力本身看不到顺序。

In [ ]:
# ============================================================
# 4. 位置编码 (Positional Encoding)
# ============================================================
# 注意力没有内置的"顺序"概念——"cat sat"和"sat cat"对它来说一样的。
# 位置编码给每个位置注入一个独特的信号，让模型知道 token 的先后顺序。
#
# 公式:
#   PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
#   PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
#
# 用 sin/cos 的原因: PE(pos+k) 可以表示为 PE(pos) 的线性函数，
# 模型能学到相对位置关系。

class PositionalEncoding(nn.Module):
    def __init__(self, d_model=512, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)               # (max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)  # (max_len, 1)
        # 除数: 10000^(2i/d_model)，用 log 空间算防溢出
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * -(torch.log(torch.tensor(10000.0)) / d_model)
        )                                                # (d_model/2,)
        pe[:, 0::2] = torch.sin(position * div_term)     # 偶数位放 sin
        pe[:, 1::2] = torch.cos(position * div_term)     # 奇数位放 cos
        pe = pe.unsqueeze(0)                              # (1, max_len, d_model) 方便加 batch
        self.register_buffer('pe', pe)                    # 不是可学习参数，但随模型保存/移动

    def forward(self, x):
        # x: (B, S, d_model)，直接加上对应长度的位置编码
        return x + self.pe[:, :x.size(1), :]

In [ ]:
# ============================================================
# 5. 前馈网络 (Position-wise Feed-Forward Network)
# ============================================================
# 每个位置的 token 独立过同一个两层 MLP。注意力负责 token 间的交互，
# FFN 负责 token 内部的非线性变换。
# 公式: FFN(x) = ReLU(x·W1 + b1)·W2 + b2
# d_model → d_ff → d_model (原论文 d_ff = 2048, d_model = 512，扩 4 倍)

class FeedForward(nn.Module):
    def __init__(self, d_model=512, d_ff=2048, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),     # 先升维: 512 → 2048
            nn.ReLU(),                     # 非线性
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),      # 再降回来: 2048 → 512
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)                 # (B, S, d_model) → (B, S, d_model)

In [ ]:
# ============================================================
# 6. Encoder 层
# ============================================================
# 结构: Self-Attention → Add&Norm → FFN → Add&Norm
# Add & Norm = 残差连接 + Layer Normalization。
# 残差让梯度可以直接回流，深层网络也能训练。
# LayerNorm 在每个样本的特征维上归一化，稳定训练。

class EncoderLayer(nn.Module):
    def __init__(self, d_model=512, n_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadSelfAttention(d_model, n_heads)  # 多头自注意力
        self.ffn      = FeedForward(d_model, d_ff, dropout)         # 前馈网络
        self.norm1    = nn.LayerNorm(d_model)                        # 注意力后的 LN
        self.norm2    = nn.LayerNorm(d_model)                        # FFN 后的 LN
        self.dropout  = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # ── 子层 1: Self-Attention + 残差 + LayerNorm ──
        attn_out, _ = self.self_attn(x, mask)       # (B, S, d_model)
        x = self.norm1(x + self.dropout(attn_out))    # 残差连接: 保留原始信息

        # ── 子层 2: FFN + 残差 + LayerNorm ──
        ffn_out = self.ffn(x)                        # (B, S, d_model)
        x = self.norm2(x + self.dropout(ffn_out))     # 残差连接

        return x

In [ ]:
# ============================================================
# 6.5 通用的多头注意力 (同时支持 Self-Attention 和 Cross-Attention)
# ============================================================
# 与 MultiHeadSelfAttention 的唯一区别：Q 和 K/V 可以来自不同输入。
# 当 q_src == kv_src 时就是自注意力，q_src ≠ kv_src 时就是交叉注意力。

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)    # Q 投影
        self.W_k = nn.Linear(d_model, d_model, bias=False)    # K 投影
        self.W_v = nn.Linear(d_model, d_model, bias=False)    # V 投影
        self.W_o = nn.Linear(d_model, d_model, bias=False)    # 输出投影

    def split_heads(self, x):
        B, S, _ = x.shape
        x = x.view(B, S, self.n_heads, self.d_k)
        return x.permute(0, 2, 1, 3)

    def combine_heads(self, x):
        B, _, S, _ = x.shape
        x = x.permute(0, 2, 1, 3).contiguous()
        return x.view(B, S, self.d_model)

    def forward(self, q_src, kv_src, mask=None):
        """q_src: Q 的来源, kv_src: K/V 的来源。相同时 = 自注意力，不同时 = 交叉注意力。"""
        Q = self.split_heads(self.W_q(q_src))            # (B, heads, S_q, d_k)
        K = self.split_heads(self.W_k(kv_src))           # (B, heads, S_kv, d_k)
        V = self.split_heads(self.W_v(kv_src))           # (B, heads, S_kv, d_k)

        attn_out, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        out = self.combine_heads(attn_out)
        return self.W_o(out), attn_weights

In [ ]:
# ============================================================
# 7. Decoder 层
# ============================================================
# 结构: Masked SA → Add&Norm → Cross-Attn → Add&Norm → FFN → Add&Norm
# 比 Encoder 多一个 Cross-Attention: Q 来自 Decoder，K/V 来自 Encoder 输出。
# 第一个 Self-Attention 加 causal mask（下三角矩阵），防看到未来 token。

class DecoderLayer(nn.Module):
    def __init__(self, d_model=512, n_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, n_heads)    # Masked 自注意力
        self.cross_attn = MultiHeadAttention(d_model, n_heads)    # 交叉注意力 (Q≠KV)
        self.ffn        = FeedForward(d_model, d_ff, dropout)
        self.norm1      = nn.LayerNorm(d_model)
        self.norm2      = nn.LayerNorm(d_model)
        self.norm3      = nn.LayerNorm(d_model)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        # ── 子层 1: Masked Self-Attention ──
        # q_src=kv_src=x → 自注意力。tgt_mask 遮住未来位置。
        attn_out, _ = self.self_attn(x, x, tgt_mask)     # (B, S_tgt, d_model)
        x = self.norm1(x + self.dropout(attn_out))         # Add & Norm

        # ── 子层 2: Cross-Attention ──
        # q_src=x (decoder), kv_src=enc_output (encoder)
        # decoder 的每个位置去"查"encoder 的所有位置
        cross_out, _ = self.cross_attn(x, enc_output, src_mask)  # (B, S_tgt, d_model)
        x = self.norm2(x + self.dropout(cross_out))      # Add & Norm

        # ── 子层 3: FFN ──
        ffn_out = self.ffn(x)
        x = self.norm3(x + self.dropout(ffn_out))         # Add & Norm

        return x

In [ ]:
# ============================================================
# 8. 完整 Transformer
# ============================================================
# Encoder: Embedding + PE → N×EncoderLayer
# Decoder: Embedding + PE → N×DecoderLayer → Linear → Softmax
#
# Mask 说明:
#   src_mask: 遮住 encoder 输入中的 padding 位置 (无意义的填充 token)
#   tgt_mask: 下三角矩阵，保证 decoder 位置 i 只能看到 0..i (causal mask)

class Transformer(nn.Module):
    def __init__(self,
                 src_vocab_size, tgt_vocab_size,   # 源/目标词表大小
                 d_model=512, n_heads=8,
                 n_encoder_layers=6, n_decoder_layers=6,
                 d_ff=2048, dropout=0.1, max_len=5000):
        super().__init__()

        # ── Embedding + 位置编码 ──
        self.src_embed = nn.Embedding(src_vocab_size, d_model)       # 源语言词嵌入
        self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model)       # 目标语言词嵌入
        self.pos_enc   = PositionalEncoding(d_model, max_len)         # 位置编码 (共享)
        self.dropout   = nn.Dropout(dropout)

        # ── Encoder 堆叠 ──
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_encoder_layers)
        ])

        # ── Decoder 堆叠 ──
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_decoder_layers)
        ])

        # ── 输出头 ──
        self.output_proj = nn.Linear(d_model, tgt_vocab_size)         # d_model → 词表大小

        # 参数初始化 (Embedding 和 Linear 的权重共享一部分信息)
        self.d_model = d_model
        # 论文里的 trick: embedding 权重乘 √d_model，防止加 PE 后信号被淹没
        self.scale = d_model ** 0.5

    def encode(self, src, src_mask=None):
        """Encoder 前向: src tokens → encoder hidden states"""
        x = self.src_embed(src) * self.scale          # (B, S_src, d_model)  乘 √d_model
        x = self.pos_enc(x)                            # 加位置编码
        x = self.dropout(x)
        for layer in self.encoder_layers:
            x = layer(x, src_mask)                     # 逐层过
        return x                                       # (B, S_src, d_model)

    def decode(self, tgt, enc_output, src_mask=None, tgt_mask=None):
        """Decoder 前向: tgt tokens + encoder 输出 → decoder hidden states"""
        x = self.tgt_embed(tgt) * self.scale          # (B, S_tgt, d_model)
        x = self.pos_enc(x)                            # 加位置编码
        x = self.dropout(x)
        for layer in self.decoder_layers:
            x = layer(x, enc_output, src_mask, tgt_mask)
        return x                                       # (B, S_tgt, d_model)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        enc_output = self.encode(src, src_mask)                  # (B, S_src, d_model)
        dec_output = self.decode(tgt, enc_output, src_mask, tgt_mask)  # (B, S_tgt, d_model)
        logits = self.output_proj(dec_output)                    # (B, S_tgt, vocab_size)
        return logits

    # ---- 生成时的辅助方法 ----
    @staticmethod
    def generate_causal_mask(sz):
        """生成因果 mask (下三角矩阵)。位置 i 只能看到 0..i。"""
        return torch.tril(torch.ones(sz, sz)).unsqueeze(0).unsqueeze(0)  # (1, 1, sz, sz)

    @staticmethod
    def generate_padding_mask(seq, pad_idx=0):
        """生成 padding mask。形状: (B, 1, 1, S) → 广播后遮住 padding 位置。"""
        return (seq != pad_idx).unsqueeze(1).unsqueeze(2)      # (B, 1, 1, S)

In [ ]:
# ============================================================
# 9. 验证：小规模复制任务 (copy task)
# ============================================================
# 最简单的 seq2seq 任务：输入一串数字，输出一模一样的串。
# 目的只是验证整个 Transformer 前向+反向传播能跑通。

torch.manual_seed(42)

# 小模型，快跑
src_vocab = tgt_vocab = 20
model = Transformer(
    src_vocab_size=src_vocab, tgt_vocab_size=tgt_vocab,
    d_model=64,            # 小模型 64 维
    n_heads=4,             # 4 个头
    n_encoder_layers=2,    # 2 层 Encoder
    n_decoder_layers=2,    # 2 层 Decoder
    d_ff=128,              # FFN 128 维
    dropout=0.1,
    max_len=100,
)

# 构造数据: src = [3,7,2,9,5,1], tgt = [3,7,2,9,5,1] (复制)
src = torch.tensor([[3, 7, 2, 9, 5, 1]])              # (1, 6)
tgt = torch.tensor([[3, 7, 2, 9, 5, 1]])              # (1, 6)

# mask
tgt_mask = model.generate_causal_mask(tgt.size(1))     # (1, 1, 6, 6) 下三角

# 前向传播
logits = model(src, tgt, tgt_mask=tgt_mask)            # (1, 6, 20)

# 损失 = 预测下一个 token 的 cross-entropy
# 输入 tgt[:-1] 预测 tgt[1:]，这是标准 teacher forcing
loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits[:, :-1, :].reshape(-1, tgt_vocab), tgt[:, 1:].reshape(-1))
loss.backward()

print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")
print(f"src 形状: {src.shape}    tgt 形状: {tgt.shape}")
print(f"输出 logits 形状: {logits.shape}    (B=1, S=6, vocab=20)")
print(f"因果 mask 形状: {tgt_mask.shape}")
print(f"CrossEntropyLoss: {loss.item():.4f}   (初始 loss ~ln(20)≈3.0)")
print(f"梯度已计算: {next(model.parameters()).grad is not None}")

# 验证因果 mask: 第 i 行有 i+1 个 1
print(f"\n因果 mask (大写=可见):")
mask_show = tgt_mask[0, 0].int().tolist()
for row in mask_show:
    print(' '.join('■' if c else '□' for c in row))